# Colab Test Notebook - Product Detection Pipeline

A self-contained checker to run the pipeline in **Google Colab**.

## What this pipeline uses

**Code / config files (upload these into Colab):**
- `pipeline_utils.py` — the full pipeline (ingest → detect → dedup → match → metadata).
- `config.yaml` — every setting (models, thresholds, frame sampling, matching, output paths).
- `product_prompts.txt` — open-vocabulary product names (one name per line). YOLOE looks for these in each frame. Add or remove names here; it is a prompt list, not a fixed class list.

**Models (set in `config.yaml`, downloaded on first run):**
- **Detection:** YOLOE `yoloe-11l-seg.pt` (Ultralytics). Open-vocabulary object detector. Also uses YOLOE's built-in 4585-name vocabulary (`detection.use_builtin_vocab: true`).
- **Crop quality (optional):** Real-ESRGAN `RealESRGAN_x4plus` — AI super-resolution on product crops. If this install fails, turn `crops.super_resolution.enabled` to `false`.
- **Dedup:** CLIP `ViT-B-32` (OpenAI weights, via OpenCLIP). Embeds crops so the same product across frames is merged (`dedup.same_product_similarity: 0.92`).
- **Matching:** Google Lens via **SerpApi** (not a local model). Crops are POSTed directly to SerpApi. Keep only scores ≥ `matching.min_match_score` (0.90) from trusted ecommerce domains.

**What `config.yaml` controls (common knobs):**
- `detection.model_weights` / `detection.backend` — which detector to run.
- `detection.product_prompts_file` — path to `product_prompts.txt`.
- `detection.confidence_threshold` — drop weak boxes (default 0.35).
- `frames.sample_every_seconds` — how often to sample the video (default 1s).
- `dedup.clip_model` / `dedup.clip_pretrained` — CLIP model for merging duplicates.
- `crops.super_resolution.model` — Real-ESRGAN weights.
- `matching.min_match_score` — keep recommendations at/above 0.90.
- `matching.trusted_domains` — ecommerce allow-list (Amazon, Flipkart, etc.).

**What `product_prompts.txt` contains:** watches, bags, footwear, clothing, electronics, appliances, kitchen, furniture, beauty, sports, toys, tools, pet, auto, grocery, and more. Humans and body parts are ignored via `detection.ignore_labels` in config.

## Step-by-step process

1. **Open this notebook in Google Colab.**
2. **Turn on GPU:** Runtime → Change runtime type → Hardware accelerator: **GPU** → Save.
3. **Upload project files** into Colab (Files panel, folder icon on the left):
   - `pipeline_utils.py`
   - `config.yaml`
   - `product_prompts.txt`
4. **Install dependencies** (section 1). Colab already ships CUDA torch, so we skip reinstalling it.
5. **Optional:** install AI super-resolution (section 1b). Skip it if that cell fails; the pipeline still works.
6. **Add secrets** (key icon in the left sidebar):
   - Required: `SERPAPI_API_KEY`
   - Optional: AWS keys only if the *input video* is on a private S3 bucket
7. **Load config** and confirm `product_prompts.txt` loads.
8. **Provide a video:** upload it in the Files panel, then set `VIDEO_FILE` to that name (or use a public URL / S3 link).
9. **Run the pipeline in order:** ingest frames → detect products → dedup → preview crops → match with SerpApi (direct crop upload, **no S3**) → write metadata.
10. **Download results:** JSON + WebVTT files from the last cell.

Run every cell from top to bottom. Do not skip the install, secrets, config, or video cells.

## 1. Install dependencies
Colab already has a CUDA build of torch, so we skip reinstalling it.

In [ ]:
!pip install -q ultralytics open-clip-torch opencv-python-headless imagehash \
  yt-dlp boto3 requests google-search-results pyyaml python-dotenv tqdm

### 1b. AI super-resolution (optional but recommended for sharp crops)

`basicsr` ships a legacy `setup.py` that fails to build under Colab's modern
setuptools (the `metadata-generation-failed` / `egg_info` error). We fix it by:
1. installing `basicsr` **without build isolation** so it uses the working
   preinstalled setuptools, and
2. installing `realesrgan` with `--no-deps` so it doesn't pull the broken build.

If this cell still fails, just skip it and set `crops.super_resolution.enabled:
false` in `config.yaml` — the pipeline falls back to high-quality classic upscaling.

In [ ]:
# 1) Pin build tools that basicsr's legacy setup.py needs.
!pip install -q "setuptools<70" wheel

# 2) Install basicsr WITHOUT build isolation (uses the pinned setuptools above).
!pip install -q --no-build-isolation basicsr

# 3) Install realesrgan without deps so it can't drag in a broken basicsr build.
!pip install -q --no-deps realesrgan

# 4) Runtime patch: newer torchvision removed 'functional_tensor' which basicsr
#    imports. Shim it BEFORE importing basicsr/realesrgan anywhere.
import sys, types
try:
    import torchvision.transforms.functional_tensor  # noqa: F401
except ModuleNotFoundError:
    import torchvision.transforms.functional as _F
    _m = types.ModuleType('torchvision.transforms.functional_tensor')
    _m.rgb_to_grayscale = _F.rgb_to_grayscale
    sys.modules['torchvision.transforms.functional_tensor'] = _m
    print('Patched torchvision.transforms.functional_tensor for basicsr.')

# 5) Verify the super-resolution stack imports cleanly.
try:
    from realesrgan import RealESRGANer  # noqa: F401
    from basicsr.archs.rrdbnet_arch import RRDBNet  # noqa: F401
    print('Super-resolution ready (Real-ESRGAN).')
except Exception as e:
    print('Super-resolution NOT available:', e)
    print('-> Set crops.super_resolution.enabled: false in config.yaml to skip it.')

## 2. Upload the project files into Colab

You need these three files in the Colab working directory:
- `pipeline_utils.py`
- `config.yaml`
- `product_prompts.txt`

**How to add them:**
1. Open the **Files** panel (folder icon, left sidebar).
2. Drag the three files in, **or** run the upload cell below.
3. Confirm they appear next to this notebook, then continue.

In [ ]:
# Upload pipeline_utils.py, config.yaml, and product_prompts.txt if they are not already here.
import os
needed = ['pipeline_utils.py', 'config.yaml', 'product_prompts.txt']
missing = [f for f in needed if not os.path.exists(f)]

if missing:
    from google.colab import files
    print('Upload these files:', ', '.join(missing))
    files.upload()
else:
    print('All project files are already in the working directory.')

still_missing = [f for f in needed if not os.path.exists(f)]
assert not still_missing, f'Still missing: {still_missing}. Upload them and re-run this cell.'
!ls -la

In [ ]:
# Skip this cell. The previous cell already checks that the three files are present.

## 3. Secrets (SerpApi key; AWS only for S3 *input* videos)

Preferred: store them in Colab **Secrets** (key icon in the left sidebar).

- **Required for matching:** `SERPAPI_API_KEY`
- **Optional:** `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION` — only if you ingest a video from a private S3 bucket (option C below). Crop matching does **not** store images on S3; crops are POSTed directly to SerpApi's Image API.

This cell reads them and exports them as environment variables.

In [ ]:
import os

def _load_secret(name):
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name)

for _k in ['SERPAPI_API_KEY', 'AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY',
           'AWS_DEFAULT_REGION']:
    _v = _load_secret(_k)
    if _v:
        os.environ[_k] = _v

os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')
print('SERPAPI set:', bool(os.environ.get('SERPAPI_API_KEY')))
print('AWS creds set:', bool(os.environ.get('AWS_ACCESS_KEY_ID')))
print('Region:', os.environ.get('AWS_DEFAULT_REGION'))

## 4. Load config + confirm models and product prompts
Reads `config.yaml`, prints which models the pipeline will use, and checks that `product_prompts.txt` loaded.

In [ ]:
import importlib, pipeline_utils as pu
importlib.reload(pu)

cfg = pu.load_config('config.yaml')
prompts = pu.load_product_prompts(cfg)
print('=== Files ===')
print('Config        : config.yaml')
print('Prompts file  :', pu.get(cfg, 'detection.product_prompts_file'))
print('Prompt count  :', len(prompts))
print('First 15      :', prompts[:15])
assert len(prompts) > 50, 'Expected a large prompt list - is product_prompts.txt present?'

print('\n=== Models ===')
print('Detector      :', pu.get(cfg, 'detection.backend'), pu.get(cfg, 'detection.model_weights'))
print('Builtin vocab :', pu.get(cfg, 'detection.use_builtin_vocab'))
print('CLIP (dedup)  :', pu.get(cfg, 'dedup.clip_model'), pu.get(cfg, 'dedup.clip_pretrained'))
print('Super-res     :', pu.get(cfg, 'crops.super_resolution.enabled'),
      pu.get(cfg, 'crops.super_resolution.model'))
print('Matching      : Google Lens via SerpApi (no local model, no S3 crop upload)')

import torch
print('\n=== Runtime ===')
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Detect conf   :', pu.get(cfg, 'detection.confidence_threshold'))
print('Match score   :', pu.get(cfg, 'matching.min_match_score'))
print('Sample every  :', pu.get(cfg, 'frames.sample_every_seconds'), 's')

## 5. Provide a video

**Option A (default): add the video manually, then type its filename.**
1. In Colab, open the **Files** panel (folder icon, left sidebar).
2. Drag your video into it (or upload it), OR mount Google Drive and point to it.
3. Set `VIDEO_FILE` below to that filename/path and run the cell.

Options B and C (URL / public S3) are in the next cell if you prefer those.

In [ ]:
# OPTION A - you added the video manually; just give the file name here.
# Examples:
#   VIDEO_FILE = 'my_video.mp4'                     # file sits next to the notebook
#   VIDEO_FILE = '/content/my_video.mp4'            # Colab default working dir
#   VIDEO_FILE = '/content/drive/MyDrive/clips/x.mp4'  # from mounted Google Drive
VIDEO_FILE = 'my_video.mp4'   # <-- change to your file name

import os
assert os.path.exists(VIDEO_FILE), (
    f'File not found: {VIDEO_FILE!r}. Add the video via the Files panel (left '
    f'sidebar) and set VIDEO_FILE to its exact name/path. Files in Colab usually '
    f'live under /content/.'
)
cfg['input']['source_type'] = 'local'
cfg['input']['local_path'] = VIDEO_FILE
print('Using video:', VIDEO_FILE)

# (Optional) To use Google Drive instead, uncomment:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# OPTION B - use a public URL (uncomment).
# cfg['input']['source_type'] = 'url'
# cfg['input']['url'] = 'https://example.com/video.mp4'

# OPTION C - use a public S3 link (uncomment).
# cfg['input']['source_type'] = 's3'
# cfg['input']['s3_uri'] = 's3://my-public-bucket/videos/input.mp4'

## 6. Run the pipeline (staged, with checks)

In [ ]:
from tqdm.auto import tqdm
from collections import Counter

# --- ingest + frames ---
video_path = pu.ingest_video(cfg)
video_id = pu.slugify(video_path.stem)
frames = pu.sample_frames(video_path, cfg)
print(f'Video: {video_path}  | id={video_id}  | frames sampled={len(frames)}')
assert frames, 'No frames sampled - check the video / time window settings.'

In [ ]:
# --- detection (humans ignored) ---
detector = pu.Detector(cfg)
crops_dir = pu.get(cfg, 's3.local_crops_dir', 'output/crops')
all_detections = []
for fr in tqdm(frames, desc='Detecting'):
    for d in detector.detect_frame(fr):
        detector.save_crop(fr, d, crops_dir)
        all_detections.append(d)
print(f'{len(all_detections)} detections (humans filtered).')
print('By label:', dict(Counter(d.label for d in all_detections)))

In [ ]:
# --- dedup into distinct products ---
embedder = pu.Embedder(cfg, section='dedup')
embeddings = embedder.embed_image_paths([d.crop_path for d in all_detections])
products = pu.dedup_products(all_detections, embeddings, cfg)
print(f'{len(products)} distinct products.')
for p in products:
    print(f'  {p.product_id}: {p.label}  {p.first_seen:.1f}s->{p.last_seen:.1f}s  '
          f'({len(p.occurrences)} occ.)')

In [ ]:
# --- preview the distinct crops ---
import matplotlib.pyplot as plt
from PIL import Image
n = len(products)
if n:
    cols = min(4, n); rows = (n + cols - 1) // cols
    plt.figure(figsize=(cols*3, rows*3))
    for i, p in enumerate(products):
        plt.subplot(rows, cols, i+1)
        plt.imshow(Image.open(p.representative_crop))
        plt.title(f'{p.product_id}: {p.label}', fontsize=9); plt.axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
# --- skip S3 crop upload ---
# Google Lens matching POSTs each local crop to SerpApi's Image API and searches
# by the returned image_id (expires in ~10 minutes). Nothing is stored in S3.
cfg.setdefault('s3', {})['enabled'] = False
print('S3 crop upload disabled. Matching will send crops directly to SerpApi.')

In [ ]:
# --- match to ecommerce products (threshold + trusted domains applied) ---
matcher = pu.Matcher(cfg, embedder=embedder)
for p in tqdm(products, desc='Matching'):
    try:
        p.recommendations = matcher.match(p)  # local crop -> SerpApi Image API
    except Exception as e:
        print(f'[warn] {p.product_id} ({p.label}): {e}')
        p.recommendations = []

thr = pu.get(cfg, 'matching.min_match_score')
for p in products:
    print(f'\n{p.product_id} [{p.label}] -> {len(p.recommendations)} matches >= {thr}')
    for r in p.recommendations:
        print(f'   {r.score:.2f}  {r.title[:55]!r}  {r.price}  {r.source}')
        print(f'         {r.url}')

In [ ]:
# --- write timestamped metadata (JSON + WebVTT) ---
import cv2, json
cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS); fc = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
video_info = {
    'id': video_id, 'path': str(video_path), 'fps': fps, 'frame_count': fc,
    'width': int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0),
    'height': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0),
    'duration_seconds': (fc/fps) if fps else None,
}
cap.release()
payload = pu.write_metadata(products, video_info, cfg)
print('Wrote', pu.get(cfg, 'metadata.output_path'), 'and', pu.get(cfg, 'metadata.webvtt_path'))
print(json.dumps(payload, indent=2)[:2000])

In [ ]:
# --- download the results out of Colab ---
from google.colab import files
try:
    files.download(pu.get(cfg, 'metadata.output_path'))
    files.download(pu.get(cfg, 'metadata.webvtt_path'))
except Exception as e:
    print('Download skipped:', e)

---
**Models:** YOLOE `yoloe-11l-seg.pt` (detect) · CLIP `ViT-B-32` / OpenAI (dedup) · Real-ESRGAN `RealESRGAN_x4plus` (crops) · Google Lens via SerpApi (match).

**Tuning is all in `config.yaml`:** `matching.min_match_score` (0.90), `detection.confidence_threshold`,
`frames.sample_every_seconds`, `dedup.same_product_similarity`, `matching.trusted_domains`.
Add products by editing **`product_prompts.txt`** and re-running from step 4.